In [1]:
%pip install -q yfinance pandas numpy plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 59.0 MB/s eta 0:00:00


In [2]:

import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
import ipywidgets as widgets

from plotly.subplots import make_subplots
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=FutureWarning)

# Project output folder
PROJECT_DIR = Path("/content/Advanced_Stock_Market_Analytics")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Default tickers: US-listed stocks
TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN",
    "NVDA", "META", "TSLA"
]

START_DATE = "2022-01-01"
END_DATE = (
    pd.Timestamp.today().normalize() + pd.Timedelta(days=1)
).strftime("%Y-%m-%d")

TRADING_DAYS_PER_YEAR = 252
ROLLING_WINDOW = 20
SHORT_MA = 20
LONG_MA = 50

CHART_TEMPLATE = "plotly_dark"

print("Project initialized successfully.")
print("Tickers:", ", ".join(TICKERS))
print("Period:", START_DATE, "to", END_DATE)
print("Output folder:", PROJECT_DIR)


Project initialized successfully.
Tickers: AAPL, MSFT, GOOGL, AMZN, NVDA, META, TSLA
Period: 2022-01-01 to 2026-10-23
Output folder: /content/Advanced_Stock_Market_Analytics


In [3]:

def add_indicators(frame):
    """Calculate indicators for one ticker's time series."""

    frame = frame.sort_values("Date").copy()

    frame["MA_20"] = frame["Close"].rolling(
        SHORT_MA, min_periods=SHORT_MA
    ).mean()

    frame["MA_50"] = frame["Close"].rolling(
        LONG_MA, min_periods=LONG_MA
    ).mean()

    frame["Daily_Return"] = frame["Close"].pct_change()

    frame["Cumulative_Return"] = (
        (1 + frame["Daily_Return"].fillna(0)).cumprod() - 1
    ) * 100

    frame["Rolling_Volatility"] = (
        frame["Daily_Return"]
        .rolling(ROLLING_WINDOW, min_periods=ROLLING_WINDOW)
        .std()
        * np.sqrt(TRADING_DAYS_PER_YEAR)
        * 100
    )

    frame["Volume_MA_20"] = frame["Volume"].rolling(
        20, min_periods=20
    ).mean()

    return frame


data = (
    raw_data.groupby("Ticker", group_keys=False)
    .apply(add_indicators)
    .reset_index(drop=True)
)

data.to_csv(
    PROJECT_DIR / "stock_market_with_indicators.csv",
    index=False
)

print("Indicators calculated.")
display(data.tail(10))


NameError: name 'raw_data' is not defined

In [4]:

available_tickers = sorted(data["Ticker"].unique().tolist())

min_date = data["Date"].min().date()
max_date = data["Date"].max().date()

ticker_filter = widgets.Dropdown(
    options=available_tickers,
    value=available_tickers[0],
    description="Ticker:",
    layout=widgets.Layout(width="300px")
)

start_filter = widgets.DatePicker(
    description="Start:",
    value=min_date,
    min=min_date,
    max=max_date
)

end_filter = widgets.DatePicker(
    description="End:",
    value=max_date,
    min=min_date,
    max=max_date
)

multi_ticker_filter = widgets.SelectMultiple(
    options=available_tickers,
    value=tuple(available_tickers),
    description="Compare:",
    rows=min(7, len(available_tickers)),
    layout=widgets.Layout(width="320px")
)

display(widgets.HBox([ticker_filter]))
display(widgets.HBox([start_filter, end_filter]))
display(multi_ticker_filter)


NameError: name 'data' is not defined

In [5]:

def get_filtered_data(ticker, start, end):
    start = pd.Timestamp(start or min_date)
    end = pd.Timestamp(end or max_date)

    if start > end:
        raise ValueError("Start date must be before or equal to end date.")

    frame = data[
        (data["Ticker"] == ticker)
        & (data["Date"] >= start)
        & (data["Date"] <= end)
    ].copy()

    if frame.empty:
        raise ValueError("No data exists for the selected ticker and dates.")

    return frame


def show_price_dashboard(ticker, start, end):
    try:
        frame = get_filtered_data(ticker, start, end)
    except ValueError as error:
        print(error)
        return

    latest = frame.iloc[-1]
    first_close = frame["Close"].iloc[0]
    last_close = frame["Close"].iloc[-1]
    change_pct = (last_close / first_close - 1) * 100

    source_names = ", ".join(
        frame["DataSource"].dropna().unique()
    )

    display(HTML(f"""
    <div style="display:flex;flex-wrap:wrap;gap:12px;font-family:Arial">
      <div style="background:#111827;color:white;padding:16px;border-radius:10px;min-width:145px">
        <div style="color:#9ca3af">Latest close</div>
        <h2>{last_close:,.2f}</h2>
      </div>
      <div style="background:#111827;color:white;padding:16px;border-radius:10px;min-width:145px">
        <div style="color:#9ca3af">Period change</div>
        <h2>{change_pct:+.2f}%</h2>
      </div>
      <div style="background:#111827;color:white;padding:16px;border-radius:10px;min-width:145px">
        <div style="color:#9ca3af">Latest volume</div>
        <h2>{latest['Volume']:,.0f}</h2>
      </div>
      <div style="background:#111827;color:white;padding:16px;border-radius:10px;min-width:200px">
        <div style="color:#9ca3af">Data source</div>
        <div style="margin-top:10px">{source_names}</div>
      </div>
    </div>
    """))

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.72, 0.28],
        subplot_titles=("Candlestick & Moving Averages", "Trading Volume")
    )

    fig.add_trace(
        go.Candlestick(
            x=frame["Date"],
            open=frame["Open"],
            high=frame["High"],
            low=frame["Low"],
            close=frame["Close"],
            name="OHLC"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=frame["Date"], y=frame["MA_20"],
            name="20-day MA", line=dict(width=1.5)
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=frame["Date"], y=frame["MA_50"],
            name="50-day MA", line=dict(width=1.5)
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Bar(
            x=frame["Date"], y=frame["Volume"],
            name="Volume", opacity=0.65
        ),
        row=2, col=1
    )

    fig.update_layout(
        template=CHART_TEMPLATE,
        title=f"{ticker} — Price & Volume Analysis",
        height=780,
        xaxis_rangeslider_visible=False,
        legend=dict(orientation="h", y=1.02, x=0),
        hovermode="x unified"
    )

    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="Shares", row=2, col=1)

    fig.show()


price_output = widgets.interactive_output(
    show_price_dashboard,
    {
        "ticker": ticker_filter,
        "start": start_filter,
        "end": end_filter
    }
)

display(price_output)


NameError: name 'ticker_filter' is not defined

In [6]:

def show_returns_volatility(ticker, start, end):
    try:
        frame = get_filtered_data(ticker, start, end)
    except ValueError as error:
        print(error)
        return

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=(
            "Daily Returns (%)",
            "20-Day Rolling Annualized Volatility (%)"
        )
    )

    colors = np.where(
        frame["Daily_Return"].fillna(0) >= 0,
        "#10b981",
        "#ef4444"
    )

    fig.add_trace(
        go.Bar(
            x=frame["Date"],
            y=frame["Daily_Return"] * 100,
            marker_color=colors,
            name="Daily return"
        ),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=frame["Date"],
            y=frame["Rolling_Volatility"],
            mode="lines",
            name="Rolling volatility",
            line=dict(width=2)
        ),
        row=2, col=1
    )

    fig.update_layout(
        template=CHART_TEMPLATE,
        title=f"{ticker} — Returns & Risk Variability",
        height=700,
        hovermode="x unified"
    )

    fig.update_yaxes(title_text="Return (%)", row=1, col=1)
    fig.update_yaxes(title_text="Volatility (%)", row=2, col=1)

    fig.show()


returns_output = widgets.interactive_output(
    show_returns_volatility,
    {
        "ticker": ticker_filter,
        "start": start_filter,
        "end": end_filter
    }
)

display(returns_output)


NameError: name 'ticker_filter' is not defined

In [7]:

def show_correlation(tickers, start, end):
    if not tickers:
        print("Select at least one ticker.")
        return

    start = pd.Timestamp(start or min_date)
    end = pd.Timestamp(end or max_date)

    frame = data[
        data["Ticker"].isin(list(tickers))
        & (data["Date"] >= start)
        & (data["Date"] <= end)
    ].copy()

    returns_pivot = frame.pivot(
        index="Date",
        columns="Ticker",
        values="Daily_Return"
    )

    # Correlation requires enough overlapping observations.
    returns_pivot = returns_pivot.dropna(
        axis=1, how="all"
    )

    if returns_pivot.shape[1] == 0:
        print("No usable return data for these selections.")
        return

    corr = returns_pivot.corr(min_periods=10)

    if corr.empty:
        print("Not enough overlapping observations to calculate correlation.")
        return

    fig = px.imshow(
        corr,
        text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
        aspect="auto",
        title="Correlation Matrix — Daily Returns"
    )

    fig.update_layout(
        template=CHART_TEMPLATE,
        height=600
    )

    fig.show()


correlation_output = widgets.interactive_output(
    show_correlation,
    {
        "tickers": multi_ticker_filter,
        "start": start_filter,
        "end": end_filter
    }
)

display(correlation_output)


NameError: name 'multi_ticker_filter' is not defined

In [8]:

def show_normalized_performance(tickers, start, end):
    if not tickers:
        print("Select at least one ticker.")
        return

    start = pd.Timestamp(start or min_date)
    end = pd.Timestamp(end or max_date)

    frame = data[
        data["Ticker"].isin(list(tickers))
        & (data["Date"] >= start)
        & (data["Date"] <= end)
    ].copy()

    frame = frame.sort_values(["Ticker", "Date"])

    if frame.empty:
        print("No data available for this selection.")
        return

    frame["Indexed_Price"] = (
        frame["Close"]
        / frame.groupby("Ticker")["Close"].transform("first")
        * 100
    )

    fig = px.line(
        frame,
        x="Date",
        y="Indexed_Price",
        color="Ticker",
        title="Relative Price Performance (Start = 100)",
        hover_data={"Close": ":.2f"}
    )

    fig.add_hline(
        y=100,
        line_dash="dash",
        line_color="gray"
    )

    fig.update_layout(
        template=CHART_TEMPLATE,
        height=550,
        yaxis_title="Indexed price",
        hovermode="x unified"
    )

    fig.show()


performance_output = widgets.interactive_output(
    show_normalized_performance,
    {
        "tickers": multi_ticker_filter,
        "start": start_filter,
        "end": end_filter
    }
)

display(performance_output)


NameError: name 'multi_ticker_filter' is not defined

In [9]:

def generate_stock_insights(ticker, start, end):
    frame = get_filtered_data(ticker, start, end).copy()

    first_close = frame["Close"].iloc[0]
    last_close = frame["Close"].iloc[-1]

    period_return = (last_close / first_close - 1) * 100

    daily_returns = frame["Daily_Return"].dropna()
    average_daily_return = (
        daily_returns.mean() * 100
        if not daily_returns.empty else np.nan
    )

    realized_volatility = (
        daily_returns.std() * np.sqrt(TRADING_DAYS_PER_YEAR) * 100
        if len(daily_returns) > 1 else np.nan
    )

    max_close = frame["Close"].max()
    min_close = frame["Close"].min()

    max_drawdown = (
        frame["Close"] / frame["Close"].cummax() - 1
    ).min() * 100

    latest = frame.iloc[-1]

    insights = [
        f"Selected ticker: {ticker}.",
        f"Observations: {len(frame):,} trading rows.",
        f"Price change over the selected period: {period_return:+.2f}%.",
        f"Highest close in the selected period: {max_close:.2f}.",
        f"Lowest close in the selected period: {min_close:.2f}.",
        f"Latest 20-day moving average: "
        f"{latest['MA_20']:.2f}."
        if pd.notna(latest["MA_20"])
        else "The 20-day moving average is not available at the last observation.",
        f"Latest 50-day moving average: "
        f"{latest['MA_50']:.2f}."
        if pd.notna(latest["MA_50"])
        else "The 50-day moving average is not available at the last observation.",
        f"Annualized volatility of daily returns in the selected period: "
        f"{realized_volatility:.2f}%."
        if pd.notna(realized_volatility)
        else "Not enough returns to estimate annualized volatility.",
        f"Maximum drawdown from a previous peak within this selected window: "
        f"{max_drawdown:.2f}%.",
        f"Latest daily return: {latest['Daily_Return'] * 100:+.2f}%."
        if pd.notna(latest["Daily_Return"])
        else "Latest daily return is unavailable."
    ]

    display(HTML(
        "<h3>📊 Automatic Analysis</h3>"
        + "".join(
            f"<p style='padding:10px;border-left:3px solid #0d9488;"
            f"background:#111827;color:white;border-radius:5px'>"
            f"{item}</p>"
            for item in insights
        )
    ))

    return insights


insight_output = widgets.interactive_output(
    generate_stock_insights,
    {
        "ticker": ticker_filter,
        "start": start_filter,
        "end": end_filter
    }
)

display(insight_output)


NameError: name 'ticker_filter' is not defined

In [10]:

# Use the current filter selections
selected_ticker = ticker_filter.value
selected_start = start_filter.value or min_date
selected_end = end_filter.value or max_date

filtered_data = get_filtered_data(
    selected_ticker, selected_start, selected_end
).copy()

# Save filtered data and a separate summary
filtered_data.to_csv(
    PROJECT_DIR / "filtered_stock_data.csv",
    index=False
)

summary = filtered_data.groupby("Ticker").agg(
    observations=("Close", "count"),
    first_close=("Close", "first"),
    last_close=("Close", "last"),
    highest_close=("Close", "max"),
    lowest_close=("Close", "min"),
    average_volume=("Volume", "mean")
).reset_index()

summary["period_return_pct"] = (
    summary["last_close"] / summary["first_close"] - 1
) * 100

summary.to_csv(
    PROJECT_DIR / "stock_summary.csv",
    index=False
)

# Build a standalone candlestick chart
price_fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.72, 0.28],
    subplot_titles=("Candlestick & Moving Averages", "Volume")
)

price_fig.add_trace(
    go.Candlestick(
        x=filtered_data["Date"],
        open=filtered_data["Open"],
        high=filtered_data["High"],
        low=filtered_data["Low"],
        close=filtered_data["Close"],
        name="OHLC"
    ),
    row=1, col=1
)

for column, label in [
    ("MA_20", "20-day MA"),
    ("MA_50", "50-day MA")
]:
    price_fig.add_trace(
        go.Scatter(
            x=filtered_data["Date"],
            y=filtered_data[column],
            name=label
        ),
        row=1, col=1
    )

price_fig.add_trace(
    go.Bar(
        x=filtered_data["Date"],
        y=filtered_data["Volume"],
        name="Volume"
    ),
    row=2, col=1
)

price_fig.update_layout(
    template=CHART_TEMPLATE,
    title=f"{selected_ticker} Price & Volume",
    height=800,
    xaxis_rangeslider_visible=False
)

# Returns and volatility charts
returns_fig = go.Figure()
returns_fig.add_trace(
    go.Bar(
        x=filtered_data["Date"],
        y=filtered_data["Daily_Return"] * 100,
        name="Daily return (%)"
    )
)
returns_fig.update_layout(
    template=CHART_TEMPLATE,
    title=f"{selected_ticker} Daily Returns",
    yaxis_title="Return (%)"
)

volatility_fig = px.line(
    filtered_data,
    x="Date",
    y="Rolling_Volatility",
    title=f"{selected_ticker} Rolling Annualized Volatility"
)
volatility_fig.update_layout(
    template=CHART_TEMPLATE,
    yaxis_title="Volatility (%)"
)

# Correlation chart for all tickers over the selected date range
comparison_data = data[
    data["Date"].between(
        pd.Timestamp(selected_start),
        pd.Timestamp(selected_end)
    )
].copy()

returns_matrix = comparison_data.pivot(
    index="Date",
    columns="Ticker",
    values="Daily_Return"
)

correlation_matrix = returns_matrix.corr(min_periods=10)

correlation_fig = px.imshow(
    correlation_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Stock Daily Return Correlation"
)
correlation_fig.update_layout(template=CHART_TEMPLATE)

# Relative performance chart for the selected period
performance_data = comparison_data.sort_values(
    ["Ticker", "Date"]
).copy()

performance_data["Indexed_Price"] = (
    performance_data["Close"]
    / performance_data.groupby("Ticker")["Close"].transform("first")
    * 100
)

performance_fig = px.line(
    performance_data,
    x="Date",
    y="Indexed_Price",
    color="Ticker",
    title="Relative Stock Performance (Start = 100)"
)
performance_fig.update_layout(template=CHART_TEMPLATE)

# Export standalone HTML files
figures = {
    "price_and_volume.html": price_fig,
    "daily_returns.html": returns_fig,
    "rolling_volatility.html": volatility_fig,
    "correlation_heatmap.html": correlation_fig,
    "relative_performance.html": performance_fig
}

for filename, figure in figures.items():
    figure.write_html(
        PROJECT_DIR / filename,
        full_html=True,
        include_plotlyjs=True
    )

# Build a text report
period_return = (
    filtered_data["Close"].iloc[-1]
    / filtered_data["Close"].iloc[0] - 1
) * 100

source_summary = (
    filtered_data["DataSource"].value_counts().to_dict()
)

report = f"""
ADVANCED STOCK MARKET ANALYTICS DASHBOARD
=========================================

Selected ticker: {selected_ticker}
Selected start date: {selected_start}
Selected end date: {selected_end}
Number of observations: {len(filtered_data)}

First available close: {filtered_data['Close'].iloc[0]:.4f}
Last available close: {filtered_data['Close'].iloc[-1]:.4f}
Period price change: {period_return:+.2f}%

Highest close: {filtered_data['Close'].max():.4f}
Lowest close: {filtered_data['Close'].min():.4f}
Average volume: {filtered_data['Volume'].mean():,.0f}

Data source counts:
{source_summary}

FILES INCLUDED
--------------
stock_market_raw_data.csv
stock_market_with_indicators.csv
filtered_stock_data.csv
stock_summary.csv
price_and_volume.html
daily_returns.html
rolling_volatility.html
correlation_heatmap.html
relative_performance.html

DATA LIMITATIONS
----------------
Live prices are historical observations downloaded using yfinance.
Synthetic sample observations are explicitly labeled in the CSV files.
Moving averages and historical volatility are descriptive indicators.
Historical results do not predict future performance.

Educational project only. Not investment advice.
"""

(PROJECT_DIR / "stock_market_report.txt").write_text(
    report, encoding="utf-8"
)

# Create a landing page
html_links = "".join(
    f"<li><a href='{filename}'>{filename.replace('.html', '').replace('_', ' ').title()}</a></li>"
    for filename in figures
)

index_html = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<title>Advanced Stock Market Analytics</title>
<style>
body {{
    font-family: Arial, sans-serif;
    max-width: 900px;
    margin: 40px auto;
    padding: 0 20px;
    background: #0b1120;
    color: #f3f4f6;
}}
a {{ color: #5eead4; }}
li {{ margin: 14px 0; }}
</style>
</head>
<body>
<h1>📈 Advanced Stock Market Analytics Dashboard</h1>
<p>Ticker: {selected_ticker}</p>
<p>Period: {selected_start} to {selected_end}</p>
<h2>Interactive visualizations</h2>
<ul>{html_links}</ul>
<p>Educational use only — not investment advice.</p>
</body>
</html>
"""

(PROJECT_DIR / "index.html").write_text(
    index_html, encoding="utf-8"
)

# Package the entire project
zip_path = Path("/content/Advanced_Stock_Market_Analytics.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for file_path in PROJECT_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=str(file_path.relative_to(PROJECT_DIR))
            )

print("Export complete!")
print("Project folder:", PROJECT_DIR)
print("ZIP archive:", zip_path)
print("\nFiles included:")
for file_path in sorted(PROJECT_DIR.iterdir()):
    print("•", file_path.name)


NameError: name 'ticker_filter' is not defined

In [11]:

from google.colab import files

files.download("/content/Advanced_Stock_Market_Analytics.zip")


FileNotFoundError: Cannot find file: /content/Advanced_Stock_Market_Analytics.zip